# 01_check_inputs — Epistasis (Fluconazole / Pulvinatal)

This notebook prepares **candidate SNP sets** for PLINK epistasis tests by:
1. Loading shared genotype + SNP metadata
2. Loading drug-specific phenotype + QTL gene list
3. Expanding the QTL gene list into a clean gene table
4. Mapping genes → SNPs (from SNP annotation table)
5. Writing a PLINK `--extract` file
6. (Optional) Building candidate PLINK bfiles via `plink2`

Designed to work for both **Fluconazole** and **Pulvinatal** by editing only the CONFIG cell.

## 1) CONFIG (edit only this cell)

In [1]:
from pathlib import Path
import datetime as dt

# =========================
# EDIT ONLY THIS BLOCK
# =========================
DRUG = "PULV"        # "FLU" (Fluconazole) or "PULV" (Pulvinatal)
RUN_LABEL = "qtl"    # any short tag to separate outputs (e.g., qtl, all, v1)
TODAY = dt.date.today().strftime("%Y%m%d")

# Base directories
BASE_DIR = Path("/blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis")
DATA_DIR = BASE_DIR / "data"

# Shared genotype inputs (used only for SNP metadata sanity checks)
MARGEPI_DIR = Path("/orange/juannanzhou/MarginalEpistasis/data")
GENO_NPY     = MARGEPI_DIR / "geno_top_60.npy"
INDEX_TSV    = DATA_DIR / "index_top_60.tsv"
SNP_INFO_TSV = DATA_DIR / "snp_info_complete.tsv"

# Drug-specific inputs
PHENO_TSV = {
    "FLU":  DATA_DIR / "FLU_FMIC_fitness.tsv",
    "PULV": DATA_DIR / "PUL_FMIC_fitness.tsv",
}[DRUG]

QTL_GENES_CSV = {
    "FLU":  DATA_DIR / "Fluconazole_QTL_genes_20260122.csv",
    "PULV": DATA_DIR / "Pulvinatal_QTL_genes_20260204.csv",
}[DRUG]

# Phenotype column names
# (Pulvinatal uses ID + s; Fluconazole may differ)
PHENO_ID_COL = {"FLU": "IID", "PULV": "ID"}[DRUG]
TRAIT_COL    = {"FLU": "fitness", "PULV": "s"}[DRUG]

# =========================
# Derived paths (do not edit)
# =========================
DRUG_LONG = {"FLU": "Fluconazole", "PULV": "Pulvinatal"}[DRUG]
drug_slug = DRUG_LONG.lower()

GENES_SPLIT_TSV  = DATA_DIR / f"{DRUG_LONG}_{RUN_LABEL}_genes_split.tsv"
EXTRACT_SNPS_TXT = DATA_DIR / f"{drug_slug}_{RUN_LABEL}_candidate_snps.extract.txt"

print("=== CONFIG (01A_check_input) ===")
print("DRUG:", DRUG, "|", DRUG_LONG)
print("RUN_LABEL:", RUN_LABEL)
print("PHENO_TSV:", PHENO_TSV)
print("QTL_GENES_CSV:", QTL_GENES_CSV)
print("PHENO_ID_COL:", PHENO_ID_COL, "| TRAIT_COL:", TRAIT_COL)
print("GENO_NPY:", GENO_NPY)
print("INDEX_TSV:", INDEX_TSV)
print("SNP_INFO_TSV:", SNP_INFO_TSV)
print("GENES_SPLIT_TSV:", GENES_SPLIT_TSV)
print("EXTRACT_SNPS_TXT:", EXTRACT_SNPS_TXT)

=== CONFIG (01A_check_input) ===
DRUG: PULV | Pulvinatal
RUN_LABEL: qtl
PHENO_TSV: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/PUL_FMIC_fitness.tsv
QTL_GENES_CSV: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/Pulvinatal_QTL_genes_20260204.csv
PHENO_ID_COL: ID | TRAIT_COL: s
GENO_NPY: /orange/juannanzhou/MarginalEpistasis/data/geno_top_60.npy
INDEX_TSV: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/index_top_60.tsv
SNP_INFO_TSV: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/snp_info_complete.tsv
GENES_SPLIT_TSV: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/Pulvinatal_qtl_genes_split.tsv
EXTRACT_SNPS_TXT: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/pulvinatal_qtl_candidate_snps.extract.txt


## 2) Imports + helper utilities

In [2]:
import numpy as np
import pandas as pd

def must_exist(path: Path, label: str):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"[{label}] Missing file: {path}")
    return path

def show_df(df, n=5, title=None):
    if title:
        print(title)
    display(df.head(n))

In [3]:
import numpy as np
import pandas as pd

def must_exist(path: Path, label: str):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"[{label}] Missing file: {path}")
    return path

def show_df(df, n=5, title=None):
    if title:
        print(title)
    display(df.head(n))

## 3) Check that required files exist

In [4]:
must_exist(GENO_NPY, "GENO_NPY")
must_exist(INDEX_TSV, "INDEX_TSV")
must_exist(SNP_INFO_TSV, "SNP_INFO_TSV")
must_exist(PHENO_TSV, "PHENO_TSV")
must_exist(QTL_GENES_CSV, "QTL_GENES_CSV")

print("All required input files found!")

All required input files found!


## 4) Load genotype matrix + genotype IDs

In [5]:
G = np.load(GENO_NPY)
print("Genotype matrix shape (samples x SNPs):", G.shape)

geno_ids = pd.read_csv(INDEX_TSV, sep="\t", header=None)
geno_ids.columns = ["IID"]
print("Genotype IDs shape:", geno_ids.shape)
show_df(geno_ids, title="Genotype IDs (top rows):")

Genotype matrix shape (samples x SNPs): (59970, 41594)
Genotype IDs shape: (59970, 1)
Genotype IDs (top rows):


,IID
0,0
1,1
2,2
3,3
4,4


## 5) Load SNP metadata and confirm it matches genotype columns

In [6]:
snp = pd.read_csv(SNP_INFO_TSV, sep="\t")
print("SNP metadata shape:", snp.shape)
show_df(snp, title="SNP metadata (top rows):")

assert snp.shape[0] == G.shape[1], (
    f"Mismatch: SNP metadata rows ({snp.shape[0]}) != genotype columns ({G.shape[1]})"
)
print("SNP rows match genotype columns")

SNP metadata shape: (41594, 12)
SNP metadata (top rows):


,Unnamed: 0,Index,SNP,Chromosome,Position (bp),BY allele,RM allele,feature_type,gene_name,gene_chr_start,gene_chr_end,gene_standard_name
0,0,0,snp1,1,27210,A,G,genic,YAL063C,24000,27968,FLO9
1,1,1,snp2,1,27290,C,A,genic,YAL063C,24000,27968,FLO9
2,2,2,snp3,1,27356,T,C,genic,YAL063C,24000,27968,FLO9
3,3,3,snp4,1,27357,A,G,genic,YAL063C,24000,27968,FLO9
4,4,4,snp5,1,27370,G,A,genic,YAL063C,24000,27968,FLO9


SNP rows match genotype columns


## 6) Load QTL gene list and explode into one-gene-per-row table

Expected columns in the QTL genes CSV:
- `genes` (ORFs) and/or `genes_std` (standard gene names)
Optional (kept if present):
- `interval_id`, `chrom`, `start`, `end`, `pleio_score`, `pleio_percentile`, `lod_max`, `condition`

In [7]:
qtl = pd.read_csv(QTL_GENES_CSV)
print("QTL gene table shape:", qtl.shape)
print("QTL gene table columns:", list(qtl.columns))
show_df(qtl, title="QTL genes CSV (top rows):")

QTL gene table shape: (31, 9)
QTL gene table columns: ['interval_id', 'chrom', 'start', 'end', 'genes', 'genes_std', 'pleio_score', 'pleio_percentile', 'lod_max']
QTL genes CSV (top rows):


,interval_id,chrom,start,end,genes,genes_std,pleio_score,pleio_percentile,lod_max
0,268,1,45546,49103,"YAL054C, YAL053W, YAL051W","ACS1, FLC2, OAF1",0.035920,98.025255,174.691785
1,932,1,196564,198232,YAR042W,SWH1,0.025053,95.036271,26.065089
2,1996,2,265775,279910,"YBR013C, YBR014C, YBR015C, YBR018C, YBR019C, Y...","YBR013C, GRX7, MNN2, GAL7, GAL10, GAL1",0.007237,46.070661,39.658236
3,2260,2,343713,352302,"YBR054W, YBR055C, YBR056W, YBR056W-A, YBR057C","YRO2, PRP6, MRX18, MNC1, MUM2",0.005419,31.367544,39.094362
4,2970,2,458571,468346,"YBR108W, YBR109C, YBR109W-A, YBR110W, YBR111C,...","AIM3, CMD1, CMD1, ALG1, YSA1, SUS1, CYC8, CYC8...",0.061356,99.462654,54.995078


## 7) Build `genes_split.tsv` (exploded gene table)

In [8]:
parts = []

if "genes" in qtl.columns:
    orf_long = (
        qtl.assign(genes=qtl["genes"].astype(str).str.replace("...", "", regex=False))
           .assign(gene=lambda df: df["genes"].str.split(r",\s*", regex=True))
           .explode("gene")
    )
    orf_long["gene"] = orf_long["gene"].astype(str).str.strip().str.upper()
    orf_long = orf_long[orf_long["gene"].notna() & (orf_long["gene"] != "")]
    orf_long["gene_type"] = "orf"
    parts.append(orf_long)

if "genes_std" in qtl.columns:
    std_long = (
        qtl.assign(genes_std=qtl["genes_std"].astype(str).str.replace("...", "", regex=False))
           .assign(gene=lambda df: df["genes_std"].str.split(r",\s*", regex=True))
           .explode("gene")
    )
    std_long["gene"] = std_long["gene"].astype(str).str.strip().str.upper()
    std_long = std_long[std_long["gene"].notna() & (std_long["gene"] != "")]
    std_long["gene_type"] = "standard"
    parts.append(std_long)

assert len(parts) > 0, "Could not find 'genes' or 'genes_std' columns in QTL file."

genes_long = pd.concat(parts, ignore_index=True)

keep_base = ["interval_id", "chrom", "start", "end", "gene", "gene_type"]
extras = [c for c in ["pleio_score", "pleio_percentile", "lod_max", "condition"] if c in genes_long.columns]
genes_long = genes_long[keep_base + extras].copy()

print("Exploded gene table shape:", genes_long.shape)
print("Unique genes:", genes_long["gene"].nunique())
show_df(genes_long, n=10, title="Exploded genes (top 10 rows):")

genes_long.to_csv(GENES_SPLIT_TSV, sep="\t", index=False)
print("Wrote:", GENES_SPLIT_TSV)

Exploded gene table shape: (656, 9)
Unique genes: 601
Exploded genes (top 10 rows):


,interval_id,chrom,start,end,gene,gene_type,pleio_score,pleio_percentile,lod_max
0,268,1,45546,49103,YAL054C,orf,0.035920,98.025255,174.691785
1,268,1,45546,49103,YAL053W,orf,0.035920,98.025255,174.691785
2,268,1,45546,49103,YAL051W,orf,0.035920,98.025255,174.691785
3,932,1,196564,198232,YAR042W,orf,0.025053,95.036271,26.065089
4,1996,2,265775,279910,YBR013C,orf,0.007237,46.070661,39.658236
5,1996,2,265775,279910,YBR014C,orf,0.007237,46.070661,39.658236
6,1996,2,265775,279910,YBR015C,orf,0.007237,46.070661,39.658236
7,1996,2,265775,279910,YBR018C,orf,0.007237,46.070661,39.658236
8,1996,2,265775,279910,YBR019C,orf,0.007237,46.070661,39.658236
9,1996,2,265775,279910,YBR020W,orf,0.007237,46.070661,39.658236


Wrote: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/Pulvinatal_qtl_genes_split.tsv


## 9) Map candidate genes → SNPs using SNP annotation table and write PLINK extract list
This creates the `*.extract.txt` file used by `plink2 --extract`.

In [9]:
genes_long = pd.read_csv(GENES_SPLIT_TSV, sep="\t")
genes_long["gene"] = genes_long["gene"].astype(str).str.strip().str.upper()
candidate_genes = set(genes_long["gene"])
print("Unique candidate genes:", len(candidate_genes))

# SNP annotation columns (try both common conventions)
snp_orf = snp.get("gene_name", pd.Series([""] * len(snp))).astype(str).str.strip().str.upper()
snp_std = snp.get("gene_standard_name", pd.Series([""] * len(snp))).astype(str).str.strip().str.upper()

cand_snps = snp[snp_orf.isin(candidate_genes) | snp_std.isin(candidate_genes)].copy()

print("Candidate SNP rows:", cand_snps.shape[0])
print("Unique candidate SNP IDs:", cand_snps["SNP"].nunique())

# Write extract file (one SNP ID per line, no header)
cand_snps["SNP"].astype(str).drop_duplicates().to_csv(EXTRACT_SNPS_TXT, index=False, header=False)
print("Wrote extract list:", EXTRACT_SNPS_TXT)

# Quick peek
cols = [c for c in ["SNP", "Chromosome", "Position (bp)", "gene_name", "gene_standard_name"] if c in cand_snps.columns]
show_df(cand_snps[cols], n=12, title="Candidate SNPs (top rows):")

Unique candidate genes: 601
Candidate SNP rows: 3490
Unique candidate SNP IDs: 3490
Wrote extract list: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/pulvinatal_qtl_candidate_snps.extract.txt
Candidate SNPs (top rows):


,SNP,Chromosome,Position (bp),gene_name,gene_standard_name
238,snp239,1,43004,YAL054C,ACS1
239,snp240,1,43137,YAL054C,ACS1
240,snp241,1,43478,YAL054C,ACS1
241,snp242,1,43565,YAL054C,ACS1
242,snp243,1,43691,YAL054C,ACS1
243,snp244,1,43784,YAL054C,ACS1
244,snp245,1,44384,YAL054C,ACS1
245,snp246,1,44588,YAL054C,ACS1
246,snp247,1,44675,YAL054C,ACS1
247,snp248,1,44891,YAL054C,ACS1


## 11) Summary (what to feed into the shell scripts)

You now have:
- `EXTRACT_SNPS_TXT` → PLINK `--extract` list of candidate SNPs
- (optional) `PLINK_CAND_BFILE.*` → candidate PLINK bfiles created here

Next:
- Run `02_local_extract_candidates.sh` (if you didn’t run step 10 here)
- Run `03_run_epistasis_local.sh`
- Then proceed to the downstream analysis/figures notebooks.

In [10]:
print("=== OUTPUTS ===")
print("Exploded genes:", GENES_SPLIT_TSV)
print("PLINK extPLINKact SNP list:", EXTRACT_SNPS_TXT)

=== OUTPUTS ===
Exploded genes: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/Pulvinatal_qtl_genes_split.tsv
PLINK extPLINKact SNP list: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/pulvinatal_qtl_candidate_snps.extract.txt


## 12) Sanity checks (01A) — ready to run `01B_make_candidate_bfiles.sh`

This confirms:
- `genes_split.tsv` exists + looks sane
- `*.extract.txt` exists + SNP IDs match `snp_info_complete.tsv`
- Prints the exact command you should run next (01B)
- (Optional) Checks whether the *full* PLINK input bfiles exist in `plink_input/` (but does NOT require them)

In [12]:
from pathlib import Path
import pandas as pd

print("=== SANITY CHECKS (01A_check_input) ===")

# ---------- 1) genes_split.tsv ----------
genes_path = Path(GENES_SPLIT_TSV)
assert genes_path.exists(), f"Missing genes_split.tsv: {genes_path}"

genes_df = pd.read_csv(genes_path, sep="\t")
assert genes_df.shape[0] > 0, f"genes_split.tsv is empty: {genes_path}"
assert "gene" in genes_df.columns, "genes_split.tsv missing required column: gene"

print(f"genes_split.tsv OK: {genes_path}")
print(f"rows = {genes_df.shape[0]} | unique genes = {genes_df['gene'].nunique()}")

# ---------- 2) extract list ----------
extract_path = Path(EXTRACT_SNPS_TXT)
assert extract_path.exists(), f"Missing extract SNP list: {extract_path}"

extract_snps = pd.read_csv(extract_path, header=None, names=["SNP"], dtype=str)
extract_snps["SNP"] = extract_snps["SNP"].astype(str).str.strip()
extract_snps = extract_snps[
    (extract_snps["SNP"] != "") & extract_snps["SNP"].notna()
].drop_duplicates()

assert extract_snps.shape[0] > 0, f"Extract SNP list is empty: {extract_path}"

print(f"extract SNP list OK: {extract_path}")
print(f"unique SNPs = {extract_snps.shape[0]}")

# ---------- 3) SNP IDs should exist in SNP metadata ----------
snp_ids = set(snp["SNP"].astype(str))
missing = [x for x in extract_snps["SNP"].tolist() if x not in snp_ids]

if len(missing) == 0:
    print(f"All extract SNP IDs found in snp_info_complete.tsv ({extract_snps.shape[0]}/{extract_snps.shape[0]})")
else:
    print(f"WARNING: {len(missing)} extract SNP IDs not found in snp_info_complete.tsv")
    print("First 20 missing SNP IDs:", missing[:20])

# ---------- 4) Final summary ----------
print("\n01A outputs are complete and consistent.")
print("Files ready for 01B_make_candidate_bfiles.sh:")
print(f"  - {GENES_SPLIT_TSV}")
print(f"  - {EXTRACT_SNPS_TXT}")

print("\nNext step (run in terminal with PLINK available):")
print("  conda activate plink_epi")
print(f"  bash /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/scripts/01B_make_candidate_bfiles.sh {DRUG} {RUN_LABEL}")

=== SANITY CHECKS (01A_check_input) ===
genes_split.tsv OK: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/Pulvinatal_qtl_genes_split.tsv
rows = 656 | unique genes = 601
extract SNP list OK: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/pulvinatal_qtl_candidate_snps.extract.txt
unique SNPs = 3490
All extract SNP IDs found in snp_info_complete.tsv (3490/3490)

01A outputs are complete and consistent.
Files ready for 01B_make_candidate_bfiles.sh:
  - /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/Pulvinatal_qtl_genes_split.tsv
  - /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/pulvinatal_qtl_candidate_snps.extract.txt

Next step (run in terminal with PLINK available):
  conda activate plink_epi
  bash /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/scripts/01B_make_candidate_bfiles.sh PULV qtl
